## *From Language to Scenography: An Agentic Experiment in Spatial and Experiential Translation*  
Mostafa Zein Alabidine - Lebanese American University

This project explores what happens when an intention expressed through language becomes a spatial, visual and temporal experience. 

User inputs three qualitative states form one continuous sequence. An Interpreter writes a scenographic intention. A Designer develops the spatial plan and Blender code. An Evaluator compares rendered images with the original intention. Blender is the medium through which these interpretations become visible. The aim is an abstract experiential translation, rather than a finished architectural design.

### Install the required packages

Run the next cell to install the Python packages listed in `requirements.txt`. Keep that file and the included project folders with this notebook. Packages already installed will be reused.

Blender must be already installed separately.

In [ ]:
%pip install -r requirements.txt

### Import libraries and tools

Import the libraries and project tools, and define a separate output folder for this session.

In [ ]:
from pathlib import Path
from datetime import datetime

from IPython.display import display, Image
from langgraph.graph import StateGraph, START, END

from agentic_scenography.hosted import ChatGPTConnection
from agentic_scenography.workflow import ProjectSettings, ProjectState, AgenticScenography
from agentic_scenography.presentation import notebook_observer, show_result

RUN_FOLDER = Path.cwd() / "outputs" / datetime.now().strftime("%Y%m%d_%H%M%S")

### Connect to ChatGPT  
Run this cell and complete sign-in on the official OpenAI page that will open in your browser. 
You do not enter your password in the notebook, and sign-in credentials are stored outside the project folder.

The agent uses your account to request model responses. Requests count toward your account’s usage allowance. 

ChatGPT provides the hosted LLM for interpretation, code generation, and evaluation of preview images.

Once connected, the cell lists the models available to you.

In [ ]:
connection = ChatGPTConnection()
connection.sign_in()
available_models = connection.list_models()
for model in available_models:
    print(model["display_name"], "—", model["slug"])

### Set the project settings

Choose a model from the available list

Set the animation duration using `frames` and `fps`: duration = frames ÷ fps

Set the image dimensions using `width` and `height`

Set `samples`: higher values generally produce cleaner images but take longer to render

Set the technical repair and design revision limits to 0, 1 or 2

GPU rendering is selected automatically

If no compatible GPU is available, set `allow_cpu=True` to permit CPU rendering, which may be slower.

The current settings produce a 15-second animation

In [ ]:
settings = ProjectSettings(
    model="gpt-6-astra",
    frames=180,
    fps=12,
    width=768,
    height=512,
    samples=32,
    max_technical_repairs=2,
    max_design_revisions=1,
    gpu_backend="AUTO",
    allow_cpu=False,
)

if settings.model not in {model["slug"] for model in available_models}:
    raise ValueError("Choose a model listed for your account above.")

print("Animation:", settings.frames / settings.fps, "seconds")

### Agents’ instructions/Prompts

In [ ]:
INTERPRETER_PROMPT = """You are the Interpreter agent in an architectural experiment
called Agentic Scenography. Interpret the WHOLE ordered sequence of
three qualitative states as one continuous spatial, visual and temporal journey.
Preserve ambiguity and associative interpretation. Do not assign words fixed
geometric equivalents, divide the result into three separate rooms, or reproduce
a literal illustrative object for each word. Consider overlaps, anticipation,
pacing, contrast and perception. The scene remains abstract, not a finished building.
Write a concise scenographic intention that a Designer can construct and an
Evaluator can compare with rendered evidence. Distinguish physical effects from
artistic representations. Give a brief design rationale, not private reasoning.
Return only JSON with these fields:
intention (120–180 words), rationale (one sentence), states (three objects with
word, interpretation, visible_criteria list), transitions (two short descriptions),
overall_criteria (three concise, observable criteria). Do not write Blender code.
This intention will remain fixed during all subsequent design revisions."""

DESIGNER_PROMPT = """You are the Designer agent. Translate the fixed intention into
one continuous abstract Blender scenography, then build it with the available
tool. Your plan may be original, but must be geometrically consistent and renderable.
The camera travels through the composition over time. Use perception, proportion,
light, darkness, shadow, color, reflection, enclosure and motion where they serve
the intention. Do not use text labels as substitutes for spatial interpretation.
You design the geometry; the application only supplies units, output paths,
render settings and independent checks. Use real tool observations to repair
technical errors. Never claim a render or successful action without its tool result.
Preserve the intention. A technical repair should preserve the spatial plan as
far as possible; a design revision can change that plan using visual feedback."""

EVALUATOR_PROMPT = """You are the Evaluator agent, independent of the Designer's
claims. Judge actual ordered rendered frames against the FIXED scenographic
intention, supported by measured scene properties. First describe visible evidence,
then judge it. Do not infer visible success from persuasive prose or valid code.
Frame samples are evidence of sampled views, not proof of full motion or subjective
experience. Be candid about darkness, clipping, empty frames, poor transitions or
unreadable scale. Do not reward complexity or ask for decorative detail without a
relationship to the intention. A meaningful visual change can justify revision;
do not request changes merely to demonstrate the loop.
Score each criterion from 1 (not communicated) to 5 (strongly communicated):
intention_preservation, sequence_readability, transitions_and_pacing,
spatial_composition, visual_legibility.
Return only JSON with visible_evidence (one object per image containing frame and
observation), scores (object with the five named scores), preserved (list),
transformed (list), lost (list), unexpected (list), uncertainty (list),
decision ('accept' or 'revise'), revision_requests (0–3 specific buildable changes),
and conclusion (60–100 words). You may recommend changes, not rewrite the intention."""

### Connect the workflow

In [ ]:
def build_graph(project):
    """Connect the stages and their repair and revision routes."""
    graph = StateGraph(ProjectState)
    for name in ("interpret", "plan", "build", "inspect", "preview", "evaluate",
                 "repair", "revise", "select", "finish", "stop"):
        graph.add_node(name, getattr(project, name))
    graph.add_edge(START, "interpret")
    graph.add_edge("interpret", "plan")
    graph.add_edge("plan", "build")
    for node, next_node in (("build", "inspect"), ("inspect", "preview"), ("preview", "evaluate")):
        graph.add_conditional_edges(node, project.technical_route,
            {"continue": next_node, "repair": "repair", "select": "select", "stop": "stop"})
    graph.add_edge("repair", "build")
    graph.add_conditional_edges("evaluate", project.design_route, {"revise": "revise", "select": "select"})
    graph.add_edge("revise", "plan")
    graph.add_edge("select", "finish")
    graph.add_edge("finish", END)
    graph.add_edge("stop", END)
    return graph.compile()

In [ ]:
async def run_sequence(words, name):
    project = AgenticScenography(
        connection=connection,
        root=RUN_FOLDER / name,
        settings=settings,
        observer=notebook_observer,
        interpreter_prompt=INTERPRETER_PROMPT,
        designer_prompt=DESIGNER_PROMPT,
        evaluator_prompt=EVALUATOR_PROMPT,
    )
    workflow = build_graph(project)
    try:
        state = await workflow.ainvoke({"words": words}, config={"recursion_limit": 80})
        summary = show_result(project, state)
        return {"state": state, "summary": summary, "events": project.events,
                "model_calls": project.model_calls}
    finally:
        await project.blender.close()

# Enter your sequence and run
### Replace the placeholders with three words in order, update the output folder name, and run the cell to start the full process.

## Study 1: (Fill word 1) → (Fill word 2) → (Fill word 3)

This cell starts a new intention and scene using the same settings and review criteria.

In [ ]:
sequence_one = ['FILL WORD 1', 'Fill WORD 2', 'FILL WORD 3']
result_one = await run_sequence(sequence_one, "01_Word1_Word2_Word3")

## Study 2: (Fill word 1) → (Fill word 2) → (Fill word 3)

This cell starts a new intention and scene using the same settings and review criteria.

In [ ]:
sequence_two = ['FILL WORD 1', 'Fill WORD 2', 'FILL WORD 3']
result_two = await run_sequence(sequence_two, "02_Word1_Word2_Word3")